# Supplementary Figure S3: Pathway-Level Biological Recovery

- Figure / panels: `FigS3a`, `FigS3b`
- Inputs: pathway recovery payloads only
- Outputs: `artifacts/paper_figures/supp/FigS3_BiologyAndAblation/*`
- Pathway summary default: supplementary biology/pathway module; Norman remains the primary dataset
- Run order: run after the benchmark notebooks; this notebook depends on pathway exports only
- Dataset / model selector: edit `SELECTED_DATASET_KEYS` and `MODELS` in the parameter cell below
- Role: Supplementary biology/pathway module
- Baseline note: `scGPT` is included as a foundation-model baseline when pathway results are compared across models


In [1]:
from __future__ import annotations

import json
import importlib
import shutil
import sys
from pathlib import Path
from textwrap import fill

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis as paper_style_axis

apply_gears_paper_style(font_scale=1.0)
REQUEST_MODEL_ALIASES = {
    'genept': 'genepert',
}
PLOT_MODEL_ALIASES = {
    'trishift_nearest': 'trishift',
    'trishift': 'trishift',
    'genept': 'genepert',
}
MODEL_LABELS = {
    'trishift_nearest': 'TriShift',
    'trishift': 'TriShift',
    'biolord': 'biolord',
    'gears': 'GEARS',
    'genepert': 'GenePert',
    'scgpt': 'scGPT',
}
MODEL_COLORS = {
    'TriShift': '#0072B2',
    'biolord': '#E69F00',
    'GEARS': '#009E73',
    'GenePert': '#CC79A7',
    'scGPT': '#D55E00',
    'Truth': '#4A4A4A',
}

def canonical_request_model_name(model_name: str) -> str:
    key = str(model_name).strip().lower()
    return REQUEST_MODEL_ALIASES.get(key, key)


def canonical_plot_model_name(model_name: str) -> str:
    key = canonical_request_model_name(model_name)
    return PLOT_MODEL_ALIASES.get(key, key)


def normalize_model_requests(models: list[str]) -> list[str]:
    out = []
    for model_name in models:
        key = canonical_request_model_name(model_name)
        if key and key not in out:
            out.append(key)
    return out


def _style_axis(ax: plt.Axes, *, grid_axis: str = 'y') -> None:
    paper_style_axis(ax, grid_axis=grid_axis)
    ax.set_axisbelow(True)

from scripts.trishift.analysis import pathway_recovery
from scripts.trishift.analysis._result_adapter import resolve_result
importlib.reload(pathway_recovery)



<module 'scripts.trishift.analysis.pathway_recovery' from 'E:\\CODE\\trishift\\scripts\\trishift\\analysis\\pathway_recovery.py'>

In [2]:
ALL_DATASET_SPECS = [
    {'dataset_key': 'dixit', 'dataset_label': 'Dixit'},
    {'dataset_key': 'adamson', 'dataset_label': 'Adamson'},
    {'dataset_key': 'norman', 'dataset_label': 'Norman'},
    {'dataset_key': 'replogle_k562_essential', 'dataset_label': 'Replogle K562'},
    {'dataset_key': 'replogle_rpe1_essential', 'dataset_label': 'Replogle RPE1'},
]
AVAILABLE_DATASET_KEYS = [spec['dataset_key'] for spec in ALL_DATASET_SPECS]
SELECTED_DATASET_KEYS = ['norman']  # edit here
invalid_dataset_keys = [key for key in SELECTED_DATASET_KEYS if key not in AVAILABLE_DATASET_KEYS]
if invalid_dataset_keys:
    raise ValueError(f'Unknown dataset keys: {invalid_dataset_keys}. Available: {AVAILABLE_DATASET_KEYS}')
DATASET_SPECS = [spec for spec in ALL_DATASET_SPECS if spec['dataset_key'] in SELECTED_DATASET_KEYS]
if not DATASET_SPECS:
    raise ValueError('SELECTED_DATASET_KEYS produced an empty dataset list.')

MODELS = ['trishift_nearest', 'genepert', 'scgpt', 'gears', 'biolord']  # edit here
REQUESTED_MODELS = normalize_model_requests(MODELS)
AVAILABLE_SPLIT_IDS = [1]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS3_BiologyAndAblation'
FORCE_PATHWAY_RERUN = True  # edit here; set True only when you explicitly want to ignore cache and rerun pathway recovery
PATHWAY_FALLBACK_DIRS = {
    'adamson': [],
    'norman': [],
}
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Selected pathway datasets:', SELECTED_DATASET_KEYS)
print('Selected splits:', SPLIT_IDS)
print('Requested models (raw):', MODELS)
print('Requested models (canonical):', REQUESTED_MODELS)
print('Force pathway rerun:', FORCE_PATHWAY_RERUN)
print('Pathway fallback dirs:', {k: [str(p) for p in v] for k, v in PATHWAY_FALLBACK_DIRS.items()})



Selected pathway datasets: ['norman']
Selected splits: [1]
Requested models (raw): ['trishift_nearest', 'genepert', 'scgpt', 'gears', 'biolord']
Requested models (canonical): ['trishift_nearest', 'genepert', 'scgpt', 'gears', 'biolord']
Force pathway rerun: True
Pathway fallback dirs: {'adamson': [], 'norman': []}


In [3]:
def available_models_for_dataset(dataset_key: str) -> list[str]:
    available = []
    for model_name in REQUESTED_MODELS:
        try:
            resolved = resolve_result(dataset=dataset_key, model_name=model_name)
            if getattr(resolved, 'available_split_ids', None):
                available.append(model_name)
        except Exception:
            continue
    return available


def read_csv_if_nonempty(path: Path) -> pd.DataFrame | None:
    if (not path.exists()) or path.stat().st_size <= 5:
        return None
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    if df.empty or len(df.columns) == 0:
        return None
    return df


def load_pathway_cache(root: Path, dataset_key: str, required_models: list[str]) -> dict[str, object] | None:
    metric_df = read_csv_if_nonempty(root / 'pathway_metric_means.csv')
    enrichment_df = read_csv_if_nonempty(root / 'pathway_enrichment_all.csv')
    corr_df = read_csv_if_nonempty(root / 'pathway_nes_correlation.csv')
    meta_path = root / 'run_meta.json'
    if metric_df is None or enrichment_df is None or corr_df is None or not meta_path.exists():
        return None
    try:
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
    except Exception:
        return None
    models_used = [canonical_request_model_name(x) for x in meta.get('models_used', [])]
    meta_dataset = str(meta.get('dataset', '')).strip().lower()
    if meta_dataset and meta_dataset != str(dataset_key).strip().lower():
        return None
    if required_models and not set(required_models).issubset(set(models_used)):
        return None
    return {
        'metric_summary_df': metric_df,
        'enrichment_df': enrichment_df,
        'corr_df': corr_df,
        'meta': meta,
        'cache_root': str(root),
    }


def safe_pathway_run(dataset_key: str) -> dict[str, object]:
    out_dir = OUT_ROOT / f'pathway_{dataset_key}'
    if FORCE_PATHWAY_RERUN and out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    requested_models = list(REQUESTED_MODELS)
    available_models = available_models_for_dataset(dataset_key)
    missing_requested_models = [m for m in requested_models if m not in available_models]
    if not FORCE_PATHWAY_RERUN:
        live_cache = load_pathway_cache(out_dir, dataset_key, available_models)
        if live_cache is not None:
            return {
                'status': 'ready', 'result': live_cache, 'cached': True, 'cache_root': str(out_dir), 'fallback': False,
                'requested_models': requested_models, 'available_models': available_models, 'missing_requested_models': missing_requested_models,
            }
        for fallback_root in PATHWAY_FALLBACK_DIRS.get(dataset_key, []):
            fallback_cache = load_pathway_cache(Path(fallback_root), dataset_key, available_models)
            if fallback_cache is not None:
                return {
                    'status': 'ready', 'result': fallback_cache, 'cached': True, 'cache_root': str(fallback_root), 'fallback': True,
                    'requested_models': requested_models, 'available_models': available_models, 'missing_requested_models': missing_requested_models,
                }
    partial_model_dirs = [p for p in out_dir.iterdir() if p.is_dir() and (p / 'dataset_summary.csv').exists()] if out_dir.exists() else []
    if partial_model_dirs and not FORCE_PATHWAY_RERUN:
        return {
            'status': 'pending',
            'error': 'aggregate enrichment cache is empty or unreadable',
            'partial_cache': True,
            'cache_root': str(out_dir),
            'requested_models': requested_models,
            'available_models': available_models,
            'missing_requested_models': missing_requested_models,
        }
    try:
        result = pathway_recovery.run_pathway_recovery(
            dataset=dataset_key,
            models=available_models,
            split_ids=SPLIT_IDS,
            out_root=out_dir,
            truth_deg_mode='effect_size_non_dropout',
            pred_deg_mode='effect_size_non_dropout',
            enrichment_mode='run_if_available',
            enrichment_library='Reactome_2022',
            n_degs=20,
            space='full_gene',
        )
        return {
            'status': 'ready', 'result': result, 'cached': False, 'cache_root': str(out_dir), 'fallback': False,
            'requested_models': requested_models, 'available_models': available_models, 'missing_requested_models': missing_requested_models,
        }
    except Exception as exc:
        return {
            'status': 'pending', 'error': str(exc), 'cache_root': str(out_dir),
            'requested_models': requested_models, 'available_models': available_models, 'missing_requested_models': missing_requested_models,
        }


def latest_matching_dir(base: Path, pattern: str) -> Path | None:
    candidates = sorted(base.glob(pattern))
    return candidates[-1] if candidates else None



In [4]:
pathway_runs, metric_frames, enrichment_frames, corr_frames = [], [], [], []
pathway_results_by_dataset = {}
for spec in DATASET_SPECS:
    run = safe_pathway_run(spec['dataset_key'])
    pathway_runs.append({**spec, **run})
    if run['status'] != 'ready':
        continue
    requested_models = list(run.get('requested_models', REQUESTED_MODELS))
    available_models = list(run.get('available_models', []))
    metric_df = run['result']['metric_summary_df'].copy()
    enrichment_df = run['result']['enrichment_df'].copy()
    corr_df = run['result']['corr_df'].copy()

    if metric_df.empty or 'combined_score_pearson' not in metric_df.columns:
        fallback_enrichment = read_csv_if_nonempty(OUT_ROOT / f"pathway_enrichment_{spec['dataset_key']}.csv")
        if fallback_enrichment is not None:
            overlap_df_fallback, corr_df_fallback, hitk_df_fallback = pathway_recovery._build_overlap_tables(fallback_enrichment)
            rebuilt_metric_df = pathway_recovery._metric_summary(
                overlap_df=overlap_df_fallback,
                corr_df=corr_df_fallback,
                hitk_df=hitk_df_fallback,
            )
            if not rebuilt_metric_df.empty:
                metric_df = rebuilt_metric_df.copy()
                enrichment_df = fallback_enrichment.copy()
                corr_df = corr_df_fallback.copy()

    if 'model_name' not in metric_df.columns:
        if len(available_models) == 1:
            metric_df['model_name'] = canonical_plot_model_name(available_models[0])
    if 'model_name' in metric_df.columns:
        metric_df['model_name'] = metric_df['model_name'].astype(str).map(canonical_plot_model_name)
    if not enrichment_df.empty and 'model_name' in enrichment_df.columns:
        enrichment_df['model_name'] = enrichment_df['model_name'].astype(str).map(canonical_plot_model_name)
    if not corr_df.empty and 'model_name' in corr_df.columns:
        corr_df['model_name'] = corr_df['model_name'].astype(str).map(canonical_plot_model_name)

    metric_df['dataset_label'] = spec['dataset_label']
    if not enrichment_df.empty:
        enrichment_df['dataset_label'] = spec['dataset_label']
    if not corr_df.empty:
        corr_df['dataset_label'] = spec['dataset_label']

    used_models_meta = [canonical_request_model_name(x) for x in run['result'].get('meta', {}).get('models_used', [])] if isinstance(run['result'], dict) else []
    skipped_meta = run['result'].get('meta', {}).get('skipped_models', []) if isinstance(run['result'], dict) else []
    aggregated_models = sorted(metric_df['model_name'].astype(str).dropna().unique().tolist()) if not metric_df.empty and 'model_name' in metric_df.columns else []
    requested_plot_models = [canonical_plot_model_name(x) for x in requested_models]
    missing_aggregated_models = sorted(set(requested_plot_models) - set(aggregated_models))

    pathway_results_by_dataset[spec['dataset_key']] = {
        'spec': spec,
        'metric_df': metric_df,
        'enrichment_df': enrichment_df,
        'corr_df': corr_df,
        'cache_root': run.get('cache_root', ''),
        'fallback': bool(run.get('fallback', False)),
        'requested_models': requested_models,
        'available_models': available_models,
        'used_models_meta': used_models_meta,
        'aggregated_models': aggregated_models,
        'missing_aggregated_models': missing_aggregated_models,
        'skipped_models_meta': skipped_meta,
    }
    metric_frames.append(metric_df)
    metric_df.to_csv(OUT_ROOT / f"pathway_summary_{spec['dataset_key']}.csv", index=False, encoding='utf-8-sig')
    if not enrichment_df.empty:
        enrichment_frames.append(enrichment_df)
        enrichment_df.to_csv(OUT_ROOT / f"pathway_enrichment_{spec['dataset_key']}.csv", index=False, encoding='utf-8-sig')
    if not corr_df.empty:
        corr_frames.append(corr_df)
        corr_df.to_csv(OUT_ROOT / f"pathway_corr_{spec['dataset_key']}.csv", index=False, encoding='utf-8-sig')
pathway_summary_df = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
enrichment_all_df = pd.concat(enrichment_frames, ignore_index=True) if enrichment_frames else pd.DataFrame()
corr_all_df = pd.concat(corr_frames, ignore_index=True) if corr_frames else pd.DataFrame()
pathway_summary_df.to_csv(OUT_ROOT / 'paths3_summary_all.csv', index=False, encoding='utf-8-sig')
enrichment_all_df.to_csv(OUT_ROOT / 'paths3_enrichment_all.csv', index=False, encoding='utf-8-sig')
corr_all_df.to_csv(OUT_ROOT / 'paths3_corr_all.csv', index=False, encoding='utf-8-sig')
status_df = pd.DataFrame([
    {
        'dataset_key': row['dataset_key'],
        'dataset_label': row['dataset_label'],
        'status': row.get('status', 'unknown'),
        'cached': bool(row.get('cached', False)),
        'fallback': bool(row.get('fallback', False)),
        'cache_root': str(row.get('cache_root', '')),
        'requested_models_raw': ','.join(MODELS),
        'requested_models': ','.join(row.get('requested_models', [])),
        'available_models': ','.join(row.get('available_models', [])),
        'missing_requested_models': ','.join(row.get('missing_requested_models', [])),
        'used_models_meta': ','.join(pathway_results_by_dataset.get(row['dataset_key'], {}).get('used_models_meta', [])),
        'aggregated_models': ','.join(pathway_results_by_dataset.get(row['dataset_key'], {}).get('aggregated_models', [])),
        'missing_aggregated_models': ','.join(pathway_results_by_dataset.get(row['dataset_key'], {}).get('missing_aggregated_models', [])),
        'skipped_models_meta': json.dumps(pathway_results_by_dataset.get(row['dataset_key'], {}).get('skipped_models_meta', []), ensure_ascii=False),
        'error': str(row.get('error', '')),
    }
    for row in pathway_runs
])
status_df.to_csv(OUT_ROOT / 'paths3_run_status.csv', index=False, encoding='utf-8-sig')
adv_rows = []
for dataset_key, payload in pathway_results_by_dataset.items():
    metric_df = payload['metric_df'].copy()
    tri_df = metric_df[metric_df['model_name'].astype(str).str.contains('trishift', case=False, na=False)]
    if tri_df.empty:
        continue
    tri_row = tri_df.iloc[0]
    comp_df = metric_df[~metric_df['model_name'].astype(str).str.contains('trishift', case=False, na=False)].copy()
    if comp_df.empty:
        continue
    for metric_name in ['combined_score_pearson', 'top10_jaccard', 'hit_at_10']:
        if metric_name not in metric_df.columns:
            continue
        best_idx = comp_df[metric_name].astype(float).idxmax()
        best_row = comp_df.loc[best_idx]
        adv_rows.append({
            'dataset_key': dataset_key,
            'dataset_label': payload['spec']['dataset_label'],
            'metric_name': metric_name,
            'trishift_value': float(tri_row[metric_name]),
            'best_competitor': str(best_row['model_name']),
            'best_competitor_value': float(best_row[metric_name]),
            'trishift_minus_best': float(tri_row[metric_name]) - float(best_row[metric_name]),
        })
advantage_df = pd.DataFrame(adv_rows)
advantage_df.to_csv(OUT_ROOT / 'figs3a_dataset_advantage.csv', index=False, encoding='utf-8-sig')
print(status_df)



D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:292

D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:292

D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:292

D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:292

D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:292

...

D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
D:\conda_envs\

D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
D:\conda_envs\

D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
D:\conda_envs\biolord\lib\site-packages\numpy\lib\_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


  dataset_key dataset_label status  cached  fallback  \
0      norman        Norman  ready   False     False   

                                          cache_root  \
0  E:\CODE\trishift\artifacts\paper_figures\supp\...   

                            requested_models_raw  \
0  trishift_nearest,genepert,scgpt,gears,biolord   

                                requested_models  \
0  trishift_nearest,genepert,scgpt,gears,biolord   

                                available_models missing_requested_models  \
0  trishift_nearest,genepert,scgpt,gears,biolord                            

  used_models_meta                      aggregated_models  \
0                   biolord,gears,genepert,scgpt,trishift   

  missing_aggregated_models skipped_models_meta error  
0                                            []        


In [5]:
available_specs = [spec for spec in DATASET_SPECS if spec['dataset_key'] in pathway_results_by_dataset]
model_order = [canonical_plot_model_name(model_name) for model_name in REQUESTED_MODELS if canonical_plot_model_name(model_name) in MODEL_LABELS]
model_order = list(dict.fromkeys(model_order))

def _prepare_pathway_bar_df(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    bar_df = df.copy()
    bar_df['model_name'] = bar_df['model_name'].astype(str).map(canonical_plot_model_name)
    present_models = sorted(bar_df['model_name'].dropna().astype(str).unique().tolist())
    missing_models = [m for m in model_order if m not in present_models]
    bar_df = bar_df[bar_df['model_name'].isin(model_order)].copy()
    bar_df['display_name'] = bar_df['model_name'].map(MODEL_LABELS).fillna(bar_df['model_name'])
    bar_df['model_name'] = pd.Categorical(bar_df['model_name'], categories=model_order, ordered=True)
    return bar_df.sort_values('model_name'), missing_models

for spec in available_specs:
    bar_df, missing_models = _prepare_pathway_bar_df(pathway_results_by_dataset[spec['dataset_key']]['metric_df'])
    fig, ax = plt.subplots(1, 1, figsize=(6.8, 4.5), dpi=220)
    suffix = f" | missing: {', '.join(MODEL_LABELS.get(m, m) for m in missing_models)}" if missing_models else ''
    sns.barplot(data=bar_df, x='display_name', y='top10_overlap', hue='display_name', dodge=False, palette=MODEL_COLORS, ax=ax)
    ax.set_title(f"{spec['dataset_label']}: Top-pathway overlap{suffix}")
    ax.set_xlabel('')
    ax.set_ylabel('Top10 overlap')
    ax.tick_params(axis='x', rotation=18)
    _style_axis(ax)
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
    fig.tight_layout()
    fig.savefig(OUT_ROOT / f"figs3a_pathway_summary_{spec['dataset_key']}.png", bbox_inches='tight')
    plt.close(fig)
if not available_specs:
    fig, ax = plt.subplots(1, 1, figsize=(6.8, 4.5), dpi=220)
    ax.text(0.5, 0.5, 'No per-dataset pathway summary available', ha='center', va='center')
    ax.axis('off')
else:
    fig, axes = plt.subplots(len(available_specs), 1, figsize=(6.8, 4.2 * len(available_specs)), dpi=220, squeeze=False)
    for row_idx, spec in enumerate(available_specs):
        bar_df, missing_models = _prepare_pathway_bar_df(pathway_results_by_dataset[spec['dataset_key']]['metric_df'])
        suffix = f" | missing: {', '.join(MODEL_LABELS.get(m, m) for m in missing_models)}" if missing_models else ''
        ax = axes[row_idx, 0]
        sns.barplot(data=bar_df, x='display_name', y='top10_overlap', hue='display_name', dodge=False, palette=MODEL_COLORS, ax=ax)
        ax.set_title(f"{spec['dataset_label']}: Top-pathway overlap{suffix}")
        ax.set_xlabel('')
        ax.set_ylabel('Top10 overlap')
        ax.tick_params(axis='x', rotation=18)
        _style_axis(ax)
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()
fig.tight_layout(); fig.savefig(OUT_ROOT / 'figs3a_pathway_summary.png', bbox_inches='tight'); plt.close(fig)



In [6]:
PATHWAY_CASE_SPEC = {'dataset': 'norman', 'condition': 'MAPK1+ctrl', 'model_name': 'trishift'}
case_meta, case_plot_df = None, None
if not corr_all_df.empty and not enrichment_all_df.empty:
    dataset_key = str(PATHWAY_CASE_SPEC['dataset'])
    condition_key = str(PATHWAY_CASE_SPEC['condition'])
    model_key = str(PATHWAY_CASE_SPEC['model_name'])
    corr_match = corr_all_df[
        (corr_all_df['dataset'].astype(str) == dataset_key)
        & (corr_all_df['condition'].astype(str) == condition_key)
        & (corr_all_df['model_name'].astype(str) == model_key)
    ].copy()
    if not corr_match.empty:
        corr_row = corr_match.iloc[0]
        case_meta = {
            'dataset_label': corr_row['dataset_label'],
            'dataset': dataset_key,
            'condition': condition_key,
            'model_name': model_key,
        }
        mask_truth = (enrichment_all_df['dataset'].astype(str) == dataset_key) & (enrichment_all_df['condition'].astype(str) == condition_key) & (enrichment_all_df['model_name'].astype(str) == model_key) & (enrichment_all_df['list_type'].astype(str) == 'truth_deg20')
        mask_pred = (enrichment_all_df['dataset'].astype(str) == dataset_key) & (enrichment_all_df['condition'].astype(str) == condition_key) & (enrichment_all_df['model_name'].astype(str) == model_key) & (enrichment_all_df['list_type'].astype(str) == 'pred_deg20')
        truth_top = enrichment_all_df[mask_truth].sort_values(['adjusted_p_value', 'combined_score'], ascending=[True, False]).head(6)
        pred_top = enrichment_all_df[mask_pred].sort_values(['adjusted_p_value', 'combined_score'], ascending=[True, False]).head(6)
        case_plot_df = pd.concat([
            truth_top.assign(Group='Truth')[['term', 'combined_score', 'Group']],
            pred_top.assign(Group=MODEL_LABELS.get(model_key, model_key))[['term', 'combined_score', 'Group']],
        ], ignore_index=True)
if case_plot_df is not None and not case_plot_df.empty:
    case_plot_df['term_wrapped'] = case_plot_df['term'].astype(str).map(lambda x: fill(x, width=30))
    term_order = (
        case_plot_df.groupby('term_wrapped')['combined_score']
        .max()
        .sort_values(ascending=True)
        .index
        .tolist()
    )
    case_plot_df['term_wrapped'] = pd.Categorical(case_plot_df['term_wrapped'], categories=term_order, ordered=True)
    case_plot_df = case_plot_df.sort_values(['term_wrapped', 'Group']).reset_index(drop=True)
    case_plot_df.to_csv(OUT_ROOT / 'figs3b_pathway_case_values.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(11.9, 6.3), dpi=220)
    ax = sns.barplot(data=case_plot_df, y='term_wrapped', x='combined_score', hue='Group', palette=MODEL_COLORS, errorbar=None, saturation=0.95)
    for patch in ax.patches:
        patch.set_edgecolor('white')
        patch.set_linewidth(0.7)
    ax.set_xlabel('Combined score')
    ax.set_ylabel('')
    ax.set_title(f"Pathway case | {case_meta['condition']}", loc='left', pad=14)
    ax.legend(title='', frameon=False, loc='upper right', bbox_to_anchor=(1.0, 1.04), ncol=2)
    _style_axis(ax, grid_axis='x')
    plt.tight_layout(rect=[0.0, 0.0, 1.0, 0.94]); plt.savefig(OUT_ROOT / 'figs3b_pathway_case.png', bbox_inches='tight'); plt.close()
    print(case_meta); display(case_plot_df.head(16))
else:
    print('No pathway case available. This usually means enrichment results were not produced.')


{'dataset_label': 'Norman', 'dataset': 'norman', 'condition': 'MAPK1+ctrl', 'model_name': 'trishift'}


,term,combined_score,Group,term_wrapped
0,Cytokine Signaling In Immune System R-HSA-1280215,10.168760,TriShift,Cytokine Signaling In Immune\nSystem R-HSA-128...
1,Cytokine Signaling In Immune System R-HSA-1280215,10.168760,Truth,Cytokine Signaling In Immune\nSystem R-HSA-128...
2,Hemostasis R-HSA-109582,14.909137,TriShift,Hemostasis R-HSA-109582
3,Hemostasis R-HSA-109582,14.909137,Truth,Hemostasis R-HSA-109582
4,G Alpha (I) Signaling Events R-HSA-418594,15.966313,TriShift,G Alpha (I) Signaling Events\nR-HSA-418594
5,G Alpha (I) Signaling Events R-HSA-418594,15.966313,Truth,G Alpha (I) Signaling Events\nR-HSA-418594
6,Golgi Associated Vesicle Biogenesis R-HSA-432722,50.996692,TriShift,Golgi Associated Vesicle\nBiogenesis R-HSA-432722
7,Golgi Associated Vesicle Biogenesis R-HSA-432722,50.996692,Truth,Golgi Associated Vesicle\nBiogenesis R-HSA-432722
8,Factors Involved In Megakaryocyte Development ...,59.143371,TriShift,Factors Involved In\nMegakaryocyte Development...
9,Factors Involved In Megakaryocyte Development ...,59.143371,Truth,Factors Involved In\nMegakaryocyte Development...


In [7]:
print('FigS3c targeted pathway ablation is intentionally omitted from the current supplementary plan.')


FigS3c targeted pathway ablation is intentionally omitted from the current supplementary plan.


In [8]:
print('FigS3d pathway ablation is intentionally omitted from the current supplementary plan.')
print(OUT_ROOT)


FigS3d pathway ablation is intentionally omitted from the current supplementary plan.
E:\CODE\trishift\artifacts\paper_figures\supp\FigS3_BiologyAndAblation
